## No Proximal CRE transcripts analysis

- public transcriptome (bulk RNA-sequencing data) で選抜した transcripts と、[`fanta.bio`](https://fanta.bio/) を対応付けたとき、TSS ± 500 bp に CRE がない transcript がどのような状態なのかを調べるための notebook

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import matplotlib.pyplot as plt

from IPython.display import display


pl.Config.set_fmt_str_lengths(100)
os.environ["POLARS_MAX_THREADS"] = "16"

In [2]:
up5p_no_proximal_cre_transcripts = pl.read_parquet(
    "../../out/fantabio/no_proximal_cre_transcirpt_list/up5p_transcript_no_cre_v261001.parquet"
)

print(up5p_no_proximal_cre_transcripts.select("transcript_id").n_unique())
display(up5p_no_proximal_cre_transcripts)

4658


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32
"""ENST00000450744.1""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000232804.5""","[""basic""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000391555.1""",320,16,70,304,200,9,42,191,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000212866.6""","[""basic""]","""+""",1,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000626866.2""",316,13,77,303,211,3,37,208,"""HSPH1""","""HGNC:16969""","""retained_intron""","""ENSG00000120694.21""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000738043.1""",312,9,85,303,195,4,52,191,null,null,"""lncRNA""","""ENSG00000296306.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""-""",4,"""URS0002A403D1""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1
"""ENST00000400040.1""",309,12,85,297,202,5,44,197,"""HSPA1A""","""HGNC:5232""","""protein_coding""","""ENSG00000215328.6""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000487173.5""",121,77,208,44,64,52,135,12,"""PLAA""","""HGNC:9043""","""protein_coding""","""ENSG00000137055.16""","[""mRNA_start_NF"", ""cds_start_NF""]","""-""",2,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0
"""ENST00000797768.1""",92,48,266,44,40,29,182,11,null,null,"""lncRNA""","""ENSG00000303875.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""+""",4,"""URS0002A4B8C9""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1
"""ENST00000460240.5""",93,49,264,44,37,29,185,8,"""BRD2""","""HGNC:1103""","""retained_intron""","""ENSG00000215077.10""",[],"""+""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0


In [3]:
# https://data.fanta.bio/cre/v1.2.1/00README_v1.2.1.txt
fantabio_bed = pl.read_csv(
    "../../Data/Database/fantabio/human-CREv1.2.1.hg38.cre-peaks.bed",
    separator="\t",
    has_header=False,
    new_columns=[
        "cre_chrom",
        "cre_chrom_start",
        "cre_chrom_end",
        "cre_id",
        "score",
        "strand",
        "thickStart",
        "thickEnd",
        "itemRgb",
        "cre_name",
        "pooled_cre_scores"
    ]
).with_columns(
    pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
    pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
).select(
    "cre_id",
    "cre_name",
    "fantabio_cre_class"
)

display(fantabio_bed)

cre_id,cre_name,fantabio_cre_class
str,str,str
"""FCHS_1""","""cp1@DDX11L1""","""PLA"""
"""FCHS_2""","""cp1@WASH7P""","""ELA"""
"""FCHS_447416""","""cp1@DDX11L17""","""ELA"""
"""FCHS_447417""","""FCHS_447417""","""ELA"""
"""FCHS_3""","""cp2@LOC101928626""","""ELA"""
…,…,…
"""FCHS_447311""","""FCHS_447311""","""ELA"""
"""FCHS_447312""","""FCHS_447312""","""ELA"""
"""FCHS_447313""","""FCHS_447313""","""ELA"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## `fanta.bio` (ver.1.2.0) gene level CRE association

- 粕川先生から、gene - CRE level relationship file をいただいた
- これは、ver.1.2.0 に該当する
- 10 kb までの対応

## gene–CRE distance の 100 bp ビン分布

- 各行は unique な `gene_id`–`cre_id` ペア（1,062,998 行 = 1,062,998 ペア）
- `distance` を 100 bp の左閉右開ビンに分割する: `[0, 100)`, `[100, 200)`, …, `[9900, 10000)`。`distance == 10000` は最後のビン
- 縦軸は、そのビンに入った gene–CRE ペア数


In [4]:
human_gene_cre_association = pl.read_csv(
    "../../Data/Database/fantabio/gene_neighbor/human_gene_cre_associate.csv.gz",
    separator=",",
    n_threads=16
).sort("distance").join(
    fantabio_bed,
    on="cre_id",
    how="inner"
)

display(human_gene_cre_association)

gene_id,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class
str,str,i64,str,str,str,str
"""ENSG00000310526.1""","""FCHS_2""",0,"""human""","""hg38""","""cp1@WASH7P""","""ELA"""
"""ENSG00000243485.6""","""FCHS_2""",0,"""human""","""hg38""","""cp1@WASH7P""","""ELA"""
"""ENSG00000310528.1""","""FCHS_447416""",0,"""human""","""hg38""","""cp1@DDX11L17""","""ELA"""
"""ENSG00000308415.1""","""FCHS_447416""",0,"""human""","""hg38""","""cp1@DDX11L17""","""ELA"""
"""ENSG00000241860.8""","""FCHS_447416""",0,"""human""","""hg38""","""cp1@DDX11L17""","""ELA"""
…,…,…,…,…,…,…
"""ENSG00000298375.1""","""FCHS_253394""",10000,"""human""","""hg38""","""FCHS_253394""","""ELA"""
"""ENSG00000232687.1""","""FCHS_533966""",10000,"""human""","""hg38""","""FCHS_533966""","""ELA"""
"""ENSG00000302188.1""","""FCHS_413906""",10000,"""human""","""hg38""","""FCHS_413906""","""ELA"""


In [5]:
BIN_SIZE_BP = 100
MAX_DISTANCE_BP = 10_000

# Assign each distance to a 100 bp bin: [0,100), [100,200), ..., [9900,10000]
distance_binned = human_gene_cre_association.with_columns(
    pl.when(pl.col("distance") == MAX_DISTANCE_BP)
    .then(MAX_DISTANCE_BP - BIN_SIZE_BP)
    .otherwise((pl.col("distance") // BIN_SIZE_BP) * BIN_SIZE_BP)
    .alias("distance_bin_start")
)

count_df = (
    distance_binned.group_by("distance_bin_start", "fantabio_cre_class")
    .agg(pl.len().alias("count"))
    .sort("distance_bin_start", "fantabio_cre_class")
)

# Keep all 100 bp bins and CRE classes even when count is zero
all_distance_bins = pl.DataFrame(
    {"distance_bin_start": range(0, MAX_DISTANCE_BP, BIN_SIZE_BP)}
)
cre_classes = (
    human_gene_cre_association["fantabio_cre_class"]
    .unique()
    .sort()
    .to_list()
)
all_bins_classes = all_distance_bins.join(
    pl.DataFrame({"fantabio_cre_class": cre_classes}),
    how="cross",
)

gene_cre_distance_bin_counts = (
    all_bins_classes.join(
        count_df,
        on=["distance_bin_start", "fantabio_cre_class"],
        how="left",
    )
    .with_columns(pl.col("count").fill_null(0).cast(pl.Int64))
    .with_columns(
        pl.format(
            "{}-{}",
            pl.col("distance_bin_start"),
            pl.col("distance_bin_start") + BIN_SIZE_BP,
        ).alias("distance_bin")
    )
    .sort("distance_bin_start", "fantabio_cre_class")
)

display(gene_cre_distance_bin_counts)

# Pivot for stacked bar plot (style matched to 02_2_HNscore_up_transcript_fantabio.ipynb)
plot_df = gene_cre_distance_bin_counts.to_pandas()
bin_labels = [
    f"{b}-{b + BIN_SIZE_BP}" for b in range(0, MAX_DISTANCE_BP, BIN_SIZE_BP)
]

pivot_df = (
    plot_df.pivot(index="distance_bin", columns="fantabio_cre_class", values="count")
    .fillna(0)
    .astype(int)
    .reindex(bin_labels)
)

classes = pivot_df.columns.tolist()
palette = sns.color_palette("Set2", n_colors=len(classes))
color_map = dict(zip(classes, palette))

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(16, 8))

bottom = np.zeros(len(pivot_df))
x = np.arange(len(pivot_df.index))

for cls in classes:
    heights = pivot_df[cls].to_numpy()
    ax.bar(
        x,
        heights,
        bottom=bottom,
        label=cls,
        color=color_map[cls],
        edgecolor="white",
        linewidth=0.5,
    )
    bottom += heights

# Show x-axis labels every 1 kb; use bin ranges (e.g. 0-100) so bars are not read as exact distances
TICK_STEP_BINS = 10
tick_idx = np.arange(0, len(pivot_df), TICK_STEP_BINS)

ax.set_xticks(x[tick_idx])
ax.set_xticklabels(
    [bin_labels[i] for i in tick_idx],
    fontsize=11,
    rotation=30,
    ha="right",
)
ax.set_xlabel("Distance bin (bp)", fontsize=18)
ax.set_ylabel("Count (gene–CRE pairs)", fontsize=18)
ax.set_title(
    "Gene–CRE association distance distribution (100 bp bins, up to 10 kb)",
    fontsize=18,
)
ax.legend(title="fantabio CRE class", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="y", labelsize=14)

plt.tight_layout()
plt.show()

distance_bin_start,fantabio_cre_class,count,distance_bin
i64,str,i64,str
0,"""ELA""",483514,"""0-100"""
0,"""PLA""",103106,"""0-100"""
100,"""ELA""",6220,"""100-200"""
100,"""PLA""",1698,"""100-200"""
200,"""ELA""",6083,"""200-300"""
…,…,…,…
9700,"""PLA""",758,"""9700-9800"""
9800,"""ELA""",3467,"""9800-9900"""
9800,"""PLA""",753,"""9800-9900"""


<Figure size 1600x800 with 1 Axes>

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## no proximal CRE transcirpts

- TSS ± 500 bp にない transcripts が、遺伝子レベルではどの範囲に CRE があるのか調査する

In [6]:
up5p_no_proximal_cre_transcripts_gene_cre_distance = up5p_no_proximal_cre_transcripts.join(
    human_gene_cre_association,
    on="gene_id",
    how="left"
).sort("distance")

print(f"up5p_no_proximal_cre_transcripts_gene_cre_distance: {up5p_no_proximal_cre_transcripts_gene_cre_distance.select('transcript_id').n_unique()}")
display(up5p_no_proximal_cre_transcripts_gene_cre_distance)

up5p_no_proximal_cre_transcripts_gene_cre_distance: 4658


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str
"""ENST00000450744.1""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000232804.5""","[""basic""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000391555.1""",320,16,70,304,200,9,42,191,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000212866.6""","[""basic""]","""+""",1,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000400040.1""",309,12,85,297,202,5,44,197,"""HSPA1A""","""HGNC:5232""","""protein_coding""","""ENSG00000215328.6""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000391548.1""",305,15,86,290,196,6,49,190,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000224501.5""","[""basic""]","""+""",1,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000445736.1""",279,25,102,254,191,9,51,182,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000231555.4""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000312087.10""",210,29,167,181,139,9,103,130,"""NSD2""","""HGNC:12766""","""nonsense_mediated_decay""","""ENSG00000109685.20""","[""CCDS""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_305398""",9999,"""human""","""hg38""","""cp2@NELFA""","""ELA"""
"""ENST00000514630.1""",108,27,271,81,75,20,156,55,null,null,"""processed_pseudogene""","""ENSG00000248466.1""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]","""-""",4,null,null,null,"""havana_homo_sapiens""",null,0,"""FCHS_308964""",9999,"""human""","""hg38""","""FCHS_308964""","""ELA"""
"""ENST00000523791.1""",122,45,239,77,89,25,137,64,null,null,"""lncRNA""","""ENSG00000253263.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,"""URS0000466E48""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""",null,0,"""FCHS_581524""",9999,"""human""","""hg38""","""cp18@MAILR""","""ELA"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. 10 kb 範囲内に CRE が紐づかない transcript

In [7]:
no_cre_10kb_transcripts = up5p_no_proximal_cre_transcripts_gene_cre_distance.filter(
    pl.col("distance").is_null()
)

print(f"no_cre_10kb_transcripts: {no_cre_10kb_transcripts.select('transcript_id').n_unique()}")
display(no_cre_10kb_transcripts)

no_cre_10kb_transcripts: 580


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str
"""ENST00000450744.1""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000232804.5""","[""basic""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000391555.1""",320,16,70,304,200,9,42,191,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000212866.6""","[""basic""]","""+""",1,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000400040.1""",309,12,85,297,202,5,44,197,"""HSPA1A""","""HGNC:5232""","""protein_coding""","""ENSG00000215328.6""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000391548.1""",305,15,86,290,196,6,49,190,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000224501.5""","[""basic""]","""+""",1,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000445736.1""",279,25,102,254,191,9,51,182,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000231555.4""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000576236.1""",58,14,334,44,32,8,211,24,"""KRT33A""","""HGNC:6450""","""protein_coding""","""ENSG00000261986.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null
"""ENST00000781552.1""",115,71,220,44,66,43,142,23,null,null,"""lncRNA""","""ENSG00000301761.1""","[""TAGENE""]","""-""",1,"""URS0002A2C0BF""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,null,null,null,null,null,null
"""ENST00000640902.1""",59,15,332,44,31,12,208,19,"""VPS53""","""HGNC:25608""","""retained_intron""","""ENSG00000283883.2""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null


&nbsp;

&nbsp;

&nbsp;

## 1-2: Haplotype derived Gene

- ハプロタイプ由来のデータかどうかを調査

In [8]:
# For more information, see https://www.gencodegenes.org/pages/biotypes.html
gencode_gtf = pl.read_csv(
    "../../Data/Database/GENCODE_v49/gencode.v49.chr_patch_hapl_scaff.annotation.gtf",
    separator='\t',
    has_header=False,
    comment_prefix='#',
    new_columns=['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attributes'],
    n_threads=16
).filter(
    pl.col('feature') == "gene"
).with_columns(
    pl.col("attributes").str.extract(r'gene_id "([^"]+)"', 1).alias("gene_id"),
    pl.col("attributes").str.extract(r'gene_name "([^"]+)"', 1).alias("gene_name"),
).select([
    pl.col("seqname"),
    pl.col("gene_id"),
    pl.col("gene_name"),
])

display(gencode_gtf)

seqname,gene_id,gene_name
str,str,str
"""chr1""","""ENSG00000290825.2""","""DDX11L16"""
"""chr1""","""ENSG00000223972.6""","""DDX11L1"""
"""chr1""","""ENSG00000310526.1""","""WASH7P"""
"""chr1""","""ENSG00000227232.6""","""WASH7P"""
"""chr1""","""ENSG00000278267.1""","""MIR6859-1"""
…,…,…
"""KI270751.1""","""ENSG00000303867.1""","""ENSG00000303867"""
"""KI270751.1""","""ENSG00000303902.1""","""ENSG00000303902"""
"""KI270751.1""","""ENSG00000306528.1""","""ENSG00000306528"""


In [9]:
no_cre_10kb_transcripts_gtf = no_cre_10kb_transcripts.join(
    gencode_gtf,
    on="gene_id",
    how="left"
)

print(no_cre_10kb_transcripts_gtf.select("seqname").to_series().unique().to_list())
display(no_cre_10kb_transcripts_gtf)

['chr6', 'MU273375.1', 'GL383546.1', 'chr21', 'KI270831.1', 'GL000220.1', 'KI270895.1', 'GL949752.1', 'GL383581.2', 'KI270813.1', 'ML143376.1', 'chrY', 'KI270877.1', 'KI270866.1', 'chr14', 'KQ983257.1', 'GL000219.1', 'KI270902.1', 'GL000205.2', 'chr19', 'GL383564.2', 'chr13', 'KI270853.1', 'GL000250.2', 'ML143352.1', 'GL000255.2', 'KI270897.1', 'KI270744.1', 'KQ090021.1', 'GL000195.1', 'MU273380.1', 'KI270816.1', 'KN538364.1', 'KI270847.1', 'chr1', 'chr17', 'chrX', 'GL000251.2', 'KN196481.1', 'MU273338.1', 'GL949747.2', 'KI270809.1', 'KI270849.1', 'KI270879.1', 'ML143359.1', 'MU273394.1', 'MU273372.1', 'KV766196.1', 'chr5', 'KZ208914.1', 'MU273395.1', 'KI270850.1', 'KI270821.1', 'KQ031389.1', 'GL949746.1', 'GL000254.2', 'chr7', 'KI270872.1', 'KI270830.1', 'chr22', 'GL000252.2', 'chr11', 'KQ458386.1', 'MU273374.1', 'chr18', 'KN196474.1', 'KI270791.1', 'chr9', 'KQ983258.1', 'MU273330.1', 'ML143358.1', 'KI270846.1', 'GL339449.2', 'chr2', 'KI270903.1', 'GL000218.1', 'chr3', 'chr8', 'chr12'

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,seqname,gene_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,str,str
"""ENST00000450744.1""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000232804.5""","[""basic""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL000250.2""","""HSPA1B"""
"""ENST00000391555.1""",320,16,70,304,200,9,42,191,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000212866.6""","[""basic""]","""+""",1,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL000255.2""","""HSPA1B"""
"""ENST00000400040.1""",309,12,85,297,202,5,44,197,"""HSPA1A""","""HGNC:5232""","""protein_coding""","""ENSG00000215328.6""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL000255.2""","""HSPA1A"""
"""ENST00000391548.1""",305,15,86,290,196,6,49,190,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000224501.5""","[""basic""]","""+""",1,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL000251.2""","""HSPA1B"""
"""ENST00000445736.1""",279,25,102,254,191,9,51,182,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000231555.4""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL000252.2""","""HSPA1B"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000576236.1""",58,14,334,44,32,8,211,24,"""KRT33A""","""HGNC:6450""","""protein_coding""","""ENSG00000261986.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL383564.2""","""KRT33A"""
"""ENST00000781552.1""",115,71,220,44,66,43,142,23,null,null,"""lncRNA""","""ENSG00000301761.1""","[""TAGENE""]","""-""",1,"""URS0002A2C0BF""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,null,null,null,null,null,null,"""GL000218.1""","""ENSG00000301761"""
"""ENST00000640902.1""",59,15,332,44,31,12,208,19,"""VPS53""","""HGNC:25608""","""retained_intron""","""ENSG00000283883.2""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""KV766196.1""","""VPS53"""


&nbsp;

&nbsp;

### 1-3: haplotype 由来

In [10]:
no_cre_10kb_transcripts_haplotype = no_cre_10kb_transcripts_gtf.filter(
    ~pl.col("seqname").str.contains("chr")
)

print(no_cre_10kb_transcripts_haplotype.select("transcript_id").n_unique())
display(no_cre_10kb_transcripts_haplotype)

293


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,seqname,gene_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,str,str
"""ENST00000450744.1""",335,13,58,322,223,4,24,219,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000232804.5""","[""basic""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL000250.2""","""HSPA1B"""
"""ENST00000391555.1""",320,16,70,304,200,9,42,191,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000212866.6""","[""basic""]","""+""",1,null,null,"[""polya_site"", ""polya_signal""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL000255.2""","""HSPA1B"""
"""ENST00000400040.1""",309,12,85,297,202,5,44,197,"""HSPA1A""","""HGNC:5232""","""protein_coding""","""ENSG00000215328.6""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL000255.2""","""HSPA1A"""
"""ENST00000391548.1""",305,15,86,290,196,6,49,190,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000224501.5""","[""basic""]","""+""",1,null,null,"[""polya_signal"", ""polya_site""]","""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL000251.2""","""HSPA1B"""
"""ENST00000445736.1""",279,25,102,254,191,9,51,182,"""HSPA1B""","""HGNC:5233""","""protein_coding""","""ENSG00000231555.4""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL000252.2""","""HSPA1B"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000576236.1""",58,14,334,44,32,8,211,24,"""KRT33A""","""HGNC:6450""","""protein_coding""","""ENSG00000261986.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""GL383564.2""","""KRT33A"""
"""ENST00000781552.1""",115,71,220,44,66,43,142,23,null,null,"""lncRNA""","""ENSG00000301761.1""","[""TAGENE""]","""-""",1,"""URS0002A2C0BF""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,null,null,null,null,null,null,"""GL000218.1""","""ENSG00000301761"""
"""ENST00000640902.1""",59,15,332,44,31,12,208,19,"""VPS53""","""HGNC:25608""","""retained_intron""","""ENSG00000283883.2""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""KV766196.1""","""VPS53"""


&nbsp;

&nbsp;

&nbsp;

### 1-4: 常染色体 + ChrX + Chr Y 由来

- 遺伝子レベルで、CREに紐づいているかどうかを調べる

In [11]:
no_cre_10kb_transcripts_chr = no_cre_10kb_transcripts_gtf.filter(
    pl.col("seqname").str.contains("chr")
)

print(no_cre_10kb_transcripts_chr.select("transcript_id").n_unique())
display(no_cre_10kb_transcripts_chr)

287


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,seqname,gene_name
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,str,str
"""ENST00000622588.2""",241,22,143,219,152,10,89,142,"""FOXD4L6""","""HGNC:31986""","""protein_coding""","""ENSG00000273514.2""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""-""",6,null,null,null,"""havana_homo_sapiens""","[""MANE_select""]",1,null,null,null,null,null,null,"""chr9""","""FOXD4L6"""
"""ENST00000362482.1""",249,40,117,209,176,20,55,156,"""RNA5S1""","""HGNC:34362""","""rRNA""","""ENSG00000199352.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""-""",3,null,null,null,"""ncrna_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr1""","""RNA5S1"""
"""ENST00000624004.1""",216,15,175,201,141,7,103,134,null,null,"""TEC""","""ENSG00000279311.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr3""","""ENSG00000279311"""
"""ENST00000601944.1""",220,20,166,200,139,8,104,131,"""ATF4P2""","""HGNC:34025""","""processed_pseudogene""","""ENSG00000273041.1""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]","""+""",4,null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,"""chrX""","""ATF4P2"""
"""ENST00000377420.1""",210,13,183,197,128,5,118,123,"""FOXD4L5""","""HGNC:18522""","""protein_coding""","""ENSG00000204779.2""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""-""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,null,null,null,null,null,null,"""chr9""","""FOXD4L5"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000365488.1""",70,26,310,44,60,21,170,39,"""RNU6-652P""","""HGNC:47615""","""snRNA""","""ENSG00000202358.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""ncrna_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr4""","""RNU6-652P"""
"""ENST00000557832.2""",106,62,238,44,60,29,162,31,null,null,"""TEC""","""ENSG00000284540.1""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr15""","""ENSG00000284540"""
"""ENST00000488371.1""",114,70,222,44,73,43,135,30,"""RPL17P38""","""HGNC:36219""","""processed_pseudogene""","""ENSG00000240441.1""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]","""+""",4,null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr12""","""RPL17P38"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1-5: 遺伝子レベルで CRE と紐づいているかチェック

In [12]:
cre_gene_ids = human_gene_cre_association.select(
    pl.col("gene_id").unique().implode()
).to_series()


no_cre_10kb_transcripts_chr_check = no_cre_10kb_transcripts_chr.with_columns(
    has_cre_gene=pl.col("gene_id").is_in(cre_gene_ids)
)

display(no_cre_10kb_transcripts_chr_check)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,seqname,gene_name,has_cre_gene
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,str,str,bool
"""ENST00000622588.2""",241,22,143,219,152,10,89,142,"""FOXD4L6""","""HGNC:31986""","""protein_coding""","""ENSG00000273514.2""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""-""",6,null,null,null,"""havana_homo_sapiens""","[""MANE_select""]",1,null,null,null,null,null,null,"""chr9""","""FOXD4L6""",false
"""ENST00000362482.1""",249,40,117,209,176,20,55,156,"""RNA5S1""","""HGNC:34362""","""rRNA""","""ENSG00000199352.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""-""",3,null,null,null,"""ncrna_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr1""","""RNA5S1""",false
"""ENST00000624004.1""",216,15,175,201,141,7,103,134,null,null,"""TEC""","""ENSG00000279311.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr3""","""ENSG00000279311""",false
"""ENST00000601944.1""",220,20,166,200,139,8,104,131,"""ATF4P2""","""HGNC:34025""","""processed_pseudogene""","""ENSG00000273041.1""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]","""+""",4,null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,"""chrX""","""ATF4P2""",false
"""ENST00000377420.1""",210,13,183,197,128,5,118,123,"""FOXD4L5""","""HGNC:18522""","""protein_coding""","""ENSG00000204779.2""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""-""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,null,null,null,null,null,null,"""chr9""","""FOXD4L5""",false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000365488.1""",70,26,310,44,60,21,170,39,"""RNU6-652P""","""HGNC:47615""","""snRNA""","""ENSG00000202358.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""ncrna_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr4""","""RNU6-652P""",false
"""ENST00000557832.2""",106,62,238,44,60,29,162,31,null,null,"""TEC""","""ENSG00000284540.1""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr15""","""ENSG00000284540""",false
"""ENST00000488371.1""",114,70,222,44,73,43,135,30,"""RPL17P38""","""HGNC:36219""","""processed_pseudogene""","""ENSG00000240441.1""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]","""+""",4,null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr12""","""RPL17P38""",false


In [13]:
no_cre_10kb_transcripts_chr_check_false = no_cre_10kb_transcripts_chr_check.filter(
    pl.col("has_cre_gene") == False
)

print(no_cre_10kb_transcripts_chr_check_false.select("transcript_id").n_unique())
display(no_cre_10kb_transcripts_chr_check_false)

287


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,seqname,gene_name,has_cre_gene
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,str,str,bool
"""ENST00000622588.2""",241,22,143,219,152,10,89,142,"""FOXD4L6""","""HGNC:31986""","""protein_coding""","""ENSG00000273514.2""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""-""",6,null,null,null,"""havana_homo_sapiens""","[""MANE_select""]",1,null,null,null,null,null,null,"""chr9""","""FOXD4L6""",false
"""ENST00000362482.1""",249,40,117,209,176,20,55,156,"""RNA5S1""","""HGNC:34362""","""rRNA""","""ENSG00000199352.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""-""",3,null,null,null,"""ncrna_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr1""","""RNA5S1""",false
"""ENST00000624004.1""",216,15,175,201,141,7,103,134,null,null,"""TEC""","""ENSG00000279311.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr3""","""ENSG00000279311""",false
"""ENST00000601944.1""",220,20,166,200,139,8,104,131,"""ATF4P2""","""HGNC:34025""","""processed_pseudogene""","""ENSG00000273041.1""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]","""+""",4,null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,"""chrX""","""ATF4P2""",false
"""ENST00000377420.1""",210,13,183,197,128,5,118,123,"""FOXD4L5""","""HGNC:18522""","""protein_coding""","""ENSG00000204779.2""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""-""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,null,null,null,null,null,null,"""chr9""","""FOXD4L5""",false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000365488.1""",70,26,310,44,60,21,170,39,"""RNU6-652P""","""HGNC:47615""","""snRNA""","""ENSG00000202358.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,null,null,null,"""ncrna_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr4""","""RNU6-652P""",false
"""ENST00000557832.2""",106,62,238,44,60,29,162,31,null,null,"""TEC""","""ENSG00000284540.1""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr15""","""ENSG00000284540""",false
"""ENST00000488371.1""",114,70,222,44,73,43,135,30,"""RPL17P38""","""HGNC:36219""","""processed_pseudogene""","""ENSG00000240441.1""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]","""+""",4,null,null,null,"""havana_homo_sapiens""",null,0,null,null,null,null,null,null,"""chr12""","""RPL17P38""",false


In [14]:
no_cre_10kb_transcripts_chr_check_true = no_cre_10kb_transcripts_chr_check.filter(
    pl.col("has_cre_gene") == True
)

print(no_cre_10kb_transcripts_chr_check_true.select("transcript_id").n_unique())
display(no_cre_10kb_transcripts_chr_check_true)

0


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class,seqname,gene_name,has_cre_gene
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str,str,str,bool


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. 10kb 以内であればCRE が紐づく transcripts

In [15]:
cre_10kb_transcripts = up5p_no_proximal_cre_transcripts_gene_cre_distance.filter(
    (pl.col("distance").is_not_null())
)

print(cre_10kb_transcripts.select("transcript_id").n_unique())
display(cre_10kb_transcripts)

4078


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str
"""ENST00000626866.2""",316,13,77,303,211,3,37,208,"""HSPH1""","""HGNC:16969""","""retained_intron""","""ENSG00000120694.21""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_110499""",0,"""human""","""hg38""","""FCHS_110499""","""ELA"""
"""ENST00000626866.2""",316,13,77,303,211,3,37,208,"""HSPH1""","""HGNC:16969""","""retained_intron""","""ENSG00000120694.21""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_110500""",0,"""human""","""hg38""","""cp5@HSPH1""","""ELA"""
"""ENST00000626866.2""",316,13,77,303,211,3,37,208,"""HSPH1""","""HGNC:16969""","""retained_intron""","""ENSG00000120694.21""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_110501""",0,"""human""","""hg38""","""cp6@HSPH1""","""ELA"""
"""ENST00000626866.2""",316,13,77,303,211,3,37,208,"""HSPH1""","""HGNC:16969""","""retained_intron""","""ENSG00000120694.21""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_110502""",0,"""human""","""hg38""","""FCHS_110502""","""ELA"""
"""ENST00000626866.2""",316,13,77,303,211,3,37,208,"""HSPH1""","""HGNC:16969""","""retained_intron""","""ENSG00000120694.21""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_110503""",0,"""human""","""hg38""","""FCHS_110503""","""ELA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000312087.10""",210,29,167,181,139,9,103,130,"""NSD2""","""HGNC:12766""","""nonsense_mediated_decay""","""ENSG00000109685.20""","[""CCDS""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_305398""",9999,"""human""","""hg38""","""cp2@NELFA""","""ELA"""
"""ENST00000514630.1""",108,27,271,81,75,20,156,55,null,null,"""processed_pseudogene""","""ENSG00000248466.1""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]","""-""",4,null,null,null,"""havana_homo_sapiens""",null,0,"""FCHS_308964""",9999,"""human""","""hg38""","""FCHS_308964""","""ELA"""
"""ENST00000523791.1""",122,45,239,77,89,25,137,64,null,null,"""lncRNA""","""ENSG00000253263.1""","[""basic"", ""Ensembl_canonical"", ""GENCODE_Primary""]","""+""",3,"""URS0000466E48""","""ENSEMBL_GENCODE""",null,"""havana_homo_sapiens""",null,0,"""FCHS_581524""",9999,"""human""","""hg38""","""cp18@MAILR""","""ELA"""


&nbsp;

&nbsp;

&nbsp;

## 2-1: 500 bp 以内の CRE に紐づく transcripts

- 混乱しやすい原因は、gene level だからである
- つまり、上記のデータフレームで、500 bp 以内の CRE に紐づくということは、既に別のtranscript が CRE と紐づいている状態を意味している

In [22]:
cre_10kb_transcripts_500bp = cre_10kb_transcripts.filter(
    pl.col("distance") <= 500
).sort(
    pl.col("HN2"),
    descending=True
)

print(cre_10kb_transcripts_500bp.select("transcript_id").n_unique())

pl.Config.set_tbl_rows(20)
print(cre_10kb_transcripts_500bp.group_by("transcript_type").agg(pl.col("transcript_id").n_unique()).sort("transcript_id", descending=True))

pl.Config.set_tbl_rows(2)
display(cre_10kb_transcripts_500bp)

3287
shape: (18, 2)
┌────────────────────────────────────┬───────────────┐
│ transcript_type                    ┆ transcript_id │
│ ---                                ┆ ---           │
│ str                                ┆ u32           │
╞════════════════════════════════════╪═══════════════╡
│ retained_intron                    ┆ 1115          │
│ protein_coding                     ┆ 658           │
│ lncRNA                             ┆ 585           │
│ protein_coding_CDS_not_defined     ┆ 562           │
│ nonsense_mediated_decay            ┆ 288           │
│ processed_pseudogene               ┆ 25            │
│ TEC                                ┆ 16            │
│ transcribed_unprocessed_pseudogene ┆ 12            │
│ IG_C_gene                          ┆ 6             │
│ snoRNA                             ┆ 5             │
│ unprocessed_pseudogene             ┆ 4             │
│ miRNA                              ┆ 3             │
│ transcribed_unitary_pseudogene     ┆ 2     

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str
"""ENST00000626866.2""",316,13,77,303,211,3,37,208,"""HSPH1""","""HGNC:16969""","""retained_intron""","""ENSG00000120694.21""",[],"""-""",0,null,null,null,"""ensembl_havana_gene_homo_sapiens""",null,0,"""FCHS_110499""",0,"""human""","""hg38""","""FCHS_110499""","""ELA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000673774.1""",69,25,312,44,32,9,210,23,"""CAPN3""","""HGNC:1480""","""retained_intron""","""ENSG00000092529.26""","[""RNA_Seq_supported_only""]","""+""",1,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""RNA-Seq supported only""]",1,"""FCHS_136185""",481,"""human""","""hg38""","""FCHS_136185""","""ELA"""


&nbsp;

&nbsp;

&nbsp;

## 2-1 (番外編): `retained_intron`

- `retained_intron` を調査する

In [43]:
cre_10kb_transcripts_500bp_retained_intron = cre_10kb_transcripts_500bp.filter(
    pl.col("transcript_type") == "retained_intron"
).select(
    "gene_id",
    "gene_symbol",
    "transcript_id",
    "HN2",
    "HN2_stranded",
    "transcript_type",
    "strand"
).unique(
    subset=["transcript_id", "HN2", "HN2_stranded", "transcript_type", "gene_id", "gene_symbol"]
).sort(
    pl.col("HN2"),
    descending=True
)

print(cre_10kb_transcripts_500bp_retained_intron.select("transcript_id").n_unique())
display(cre_10kb_transcripts_500bp_retained_intron)

1115


gene_id,gene_symbol,transcript_id,HN2,HN2_stranded,transcript_type,strand
str,str,str,i64,i64,str,str
"""ENSG00000120694.21""","""HSPH1""","""ENST00000626866.2""",303,208,"""retained_intron""","""-"""
…,…,…,…,…,…,…
"""ENSG00000172575.15""","""RASGRP1""","""ENST00000697779.1""",44,27,"""retained_intron""","""-"""


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2-2: 500 bp 以内の CREにも紐づかない transcript

- 上記の条件でもひっかからないケース

In [24]:
cre_id_already_associated = cre_10kb_transcripts_500bp.select(
    pl.col("transcript_id").unique().implode()
).to_series()

cre_10kb_transcripts_500bp_no_cre_id = cre_10kb_transcripts.filter(
    ~pl.col("transcript_id").is_in(cre_id_already_associated)
).sort(
    pl.col("HN2"),
    descending=True
)


print(cre_10kb_transcripts_500bp_no_cre_id.select("transcript_id").n_unique())

pl.Config.set_tbl_rows(20)
print(cre_10kb_transcripts_500bp_no_cre_id.group_by("transcript_type").agg(pl.col("transcript_id").n_unique()).sort("transcript_id", descending=True))

pl.Config.set_tbl_rows(2)
display(cre_10kb_transcripts_500bp_no_cre_id)

791
shape: (20, 2)
┌────────────────────────────────────┬───────────────┐
│ transcript_type                    ┆ transcript_id │
│ ---                                ┆ ---           │
│ str                                ┆ u32           │
╞════════════════════════════════════╪═══════════════╡
│ lncRNA                             ┆ 328           │
│ processed_pseudogene               ┆ 250           │
│ protein_coding                     ┆ 56            │
│ TEC                                ┆ 39            │
│ transcribed_processed_pseudogene   ┆ 27            │
│ retained_intron                    ┆ 22            │
│ misc_RNA                           ┆ 20            │
│ unprocessed_pseudogene             ┆ 15            │
│ transcribed_unprocessed_pseudogene ┆ 8             │
│ nonsense_mediated_decay            ┆ 7             │
│ miRNA                              ┆ 5             │
│ snRNA                              ┆ 4             │
│ rRNA                               ┆ 2      

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,distance,organism,genome_assembly,cre_name,fantabio_cre_class
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,i64,str,str,str,str
"""ENST00000738043.1""",312,9,85,303,195,4,52,191,null,null,"""lncRNA""","""ENSG00000296306.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""-""",4,"""URS0002A403D1""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_359461""",834,"""human""","""hg38""","""FCHS_359461""","""ELA"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENST00000542775.1""",55,11,340,44,31,3,217,28,null,null,"""processed_pseudogene""","""ENSG00000228286.3""","[""pseudo_consens"", ""basic"", … ""GENCODE_Primary""]","""-""",4,null,null,null,"""havana_homo_sapiens""",null,0,"""FCHS_76226""",9956,"""human""","""hg38""","""FCHS_76226""","""ELA"""


&nbsp;

&nbsp;

### 